In [ ]:
import os
from dotenv import load_dotenv
from modules.get_token import get_token
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
import requests
import json


def get_type_schema(token, type_name):
    """
    Fetch all available fields for a given type using GraphQL introspection
    """
    url = f"{start_url}/graphql?token={token}"
    headers = {"content-type": "application/json"}
    
    # GraphQL introspection query to get type fields
    introspection_query = f"""
    query IntrospectionQuery {{
      __type(name: "{type_name}") {{
        name
        description
        fields(includeDeprecated: false) {{
          name
          description
          type {{
            name
            kind
            ofType {{
              name
              kind
            }}
          }}
        }}
      }}
    }}
    """
    
    payload = {"query": introspection_query}
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if 'errors' in data:
            print("GraphQL Errors:")
            print(json.dumps(data['errors'], indent=2))
            return None
            
        type_info = data.get('data', {}).get('__type')
        
        if not type_info:
            print(f"{type_name} type not found in schema")
            return None
            
        return type_info
        
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None


def list_all_types(token):
    """
    List all available types in the GraphQL schema
    """
    url = f"https://msp.adionsystems.com/api/graphql?token={token}"
    headers = {"content-type": "application/json"}
    
    query = """
    query {
      __schema {
        types {
          name
          kind
          description
        }
      }
    }
    """
    
    payload = {"query": query}
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if 'errors' in data:
            print("GraphQL Errors:")
            print(json.dumps(data['errors'], indent=2))
            return []
            
        types = data.get('data', {}).get('__schema', {}).get('types', [])
        
        # Filter out introspection types and show only OBJECT types
        relevant_types = [
            t for t in types 
            if not t['name'].startswith('__') and t['kind'] == 'OBJECT'
        ]
        
        return relevant_types
        
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []


def display_fields(type_info):
    """
    Display all available fields in a readable format
    """
    if not type_info:
        return None
        
    print(f"\n{'='*80}")
    print(f"Type: {type_info['name']}")
    if type_info.get('description'):
        print(f"Description: {type_info['description']}")
    print(f"{'='*80}\n")
    
    fields = type_info.get('fields', [])
    
    if not fields:
        print("No fields found for this type.")
        return None
    
    print(f"Total Available Fields: {len(fields)}\n")
    print(f"{'Field Name':<35} {'Type':<25} {'Description'}")
    print(f"{'-'*35} {'-'*25} {'-'*40}")
    
    for field in fields:
        field_name = field['name']
        field_type = field['type']
        
        # Parse type information
        if field_type['kind'] == 'NON_NULL':
            type_name = f"{field_type['ofType']['name']}!"
        elif field_type['kind'] == 'LIST':
            type_name = f"[{field_type['ofType']['name']}]"
        else:
            type_name = field_type.get('name', 'Unknown')
        
        description = field.get('description') or ''
        description_display = description[:40] if description else ''
        
        print(f"{field_name:<35} {type_name:<25} {description_display}")
    
    return fields


def generate_sample_query(type_name, fields):
    """
    Generate a sample GraphQL query with all available fields
    """
    if not fields:
        return
        
    print(f"\n{'='*80}")
    print(f"SAMPLE QUERY WITH ALL FIELDS FOR {type_name}:")
    print(f"{'='*80}\n")
    
    # Filter out complex nested types for the basic query
    simple_fields = []
    complex_fields = []
    
    for field in fields:
        field_type = field['type']
        type_kind = field_type.get('kind', '')
        
        # Include scalar types and enums
        if type_kind in ['SCALAR', 'ENUM', 'NON_NULL']:
            simple_fields.append(field['name'])
        elif type_kind == 'OBJECT':
            complex_fields.append(f"{field['name']} {{ # OBJECT - expand as needed }}")
        elif type_kind == 'LIST':
            complex_fields.append(f"{field['name']} {{ # LIST - expand as needed }}")
    
    # Generate appropriate query based on type
    if type_name.lower() == 'invoice':
        query = f'''query {type_name.lower()}($invoiceId: String!) {{
    {type_name.lower()}(invoiceId: $invoiceId) {{
'''
    elif type_name.lower() == 'packingslip':
        query = f'''query {type_name.lower()}($packingSlipId: String!) {{
    {type_name.lower()}(packingSlipId: $packingSlipId) {{
'''
    else:
        query = f'''query {type_name.lower()}($id: String!) {{
    {type_name.lower()}(id: $id) {{
'''
    
    for field_name in simple_fields:
        query += f"        {field_name}\n"
    
    if complex_fields:
        query += "\n        # Complex fields (expand as needed):\n"
        for field_name in complex_fields:
            query += f"        # {field_name}\n"
    
    query += '''    }
}'''
    
    print(query)
    print()


def save_to_file(type_info, filename=None):
    """
    Save the schema information to a JSON file
    """
    if not type_info:
        return
        
    if not filename:
        filename = f"{type_info['name'].lower()}_fields.json"
        
    try:
        with open(filename, 'w') as f:
            json.dump(type_info, f, indent=2)
        print(f"\n✓ Schema saved to {filename}")
    except Exception as e:
        print(f"\n✗ Failed to save schema: {e}")


def main():
    token = get_token()
    if not token:
        print("Failed to get authentication token")
        return
    
    print("="*80)
    print("GraphQL Schema Inspector")
    print("="*80)
    
    # Ask user what they want to inspect
    print("\nOptions:")
    print("1. Inspect a specific type (e.g., Invoice, PackingSlip)")
    print("2. List all available types")
    print("3. Inspect multiple types")
    
    choice = input("\nEnter choice (1-3): ").strip()
    
    if choice == "2":
        print("\nFetching all available types...")
        types = list_all_types(token)
        
        print(f"\n{'='*80}")
        print(f"AVAILABLE TYPES IN SCHEMA ({len(types)} found):")
        print(f"{'='*80}\n")
        
        for i, t in enumerate(types, 1):
            desc = t.get('description', '')[:50] if t.get('description') else ''
            print(f"{i:3d}. {t['name']:<30} {desc}")
        
        print("\nTip: Use option 1 to inspect any of these types in detail")
        
    elif choice == "3":
        # Inspect multiple types
        types_to_check = ["Invoice", "PackingSlip"]
        
        for type_name in types_to_check:
            print(f"\n\nFetching {type_name} schema from GraphQL API...")
            type_info = get_type_schema(token, type_name)
            fields = display_fields(type_info)
            generate_sample_query(type_name, fields)
            save_to_file(type_info)
        
    else:  # Default to option 1
        type_name = input("\nEnter type name to inspect (e.g., Invoice, PackingSlip): ").strip()
        
        if not type_name:
            type_name = "PackingSlip"
            print(f"Using default: {type_name}")
        
        print(f"\nFetching {type_name} schema from GraphQL API...")
        type_info = get_type_schema(token, type_name)
        fields = display_fields(type_info)
        generate_sample_query(type_name, fields)
        save_to_file(type_info)


if __name__ == "__main__":
    main()